# Member B 属性预实验（开发集）
先标注、不看预测。复用已经跑通的权重及图片。本Notebook未在GPU实测，不自动运行评估集。

In [ ]:
from pathlib import Path
import subprocess, sys
code=Path('/kaggle/working/memberb-attribute')
assets=Path('/kaggle/working/team-baseline')
if not code.exists():
    subprocess.run(['git','clone','--branch','memberb','https://github.com/Ariel-tian95/Composed-Image-Retrieval.git',str(code)],check=True)
assert (code/'scripts/attribute_pilot.py').exists(), '代码不是最新版，请检查 memberb 分支'
assert (assets/'checkpoints/context_i2w/official_checkpoint.pt').exists(), '请恢复之前保存的图片和权重到 team-baseline；不用重新跑检索'
assert (assets/'external/context_i2w/model/model.py').exists()
subprocess.run([sys.executable,'-m','pip','install','ftfy','regex','einops','einops-exts','Pillow'],check=True)

## 仅预测开发集50张
模型预测不是真实标签；输出分数不是校准概率。

In [ ]:
out=Path('/kaggle/working/attribute-pilot-results')
out.mkdir(exist_ok=True)
subprocess.run([sys.executable,str(code/'scripts/attribute_pilot.py'),'predict','--root',str(assets),'--manifest',str(code/'docs/attribute-pilot/labels.csv'),'--split','dev','--out',str(out/'dev_predictions.json')],check=True)

## 上传人工导出的 labels.csv 后评价
使用右侧 Upload 添加CSV。不能上传模型预测作为标签，不能把空表当作已完成标注。

In [ ]:
import csv
candidates=list(Path('/kaggle/input').rglob('labels.csv'))
reviewed=[]
for p in candidates:
    with p.open(encoding='utf-8-sig') as f: rows=list(csv.DictReader(f))
    if any(r.get('reviewer','').strip() for r in rows): reviewed.append(p)
print('找到人工标注文件：',reviewed)
if len(reviewed)==1:
    subprocess.run([sys.executable,str(code/'scripts/attribute_pilot.py'),'evaluate','--labels',str(reviewed[0]),'--predictions',str(out/'dev_predictions.json'),'--split','dev','--out',str(out/'dev_metrics.json')],check=True)
else:
    print('暂不计算指标。请上传唯一一份人工标注CSV；若有多份请明确选择，避免误用。')

## 保存和交接
从右侧 Output 下载 dev_predictions.json 和 dev_metrics.json，保存Notebook版本。仅在开发集规则、提示词和拒识策略冻结后，才另行运行 --split eval，并且不得根据评估结果反复调参。